# 00. Metodologia Badania - BERT Triage Model

## 📋 Cel Notebooka

Ten notebook zawiera metodologię badania dla modelu BERT do klasyfikacji triage pacjentów:

1. **Table 1: Charakterystyka danych** - rozkład klas, statystyki demograficzne
2. **Flowchart przetwarzania danych** - wizualizacja pipeline od surowych danych do modelu
3. **Data Dictionary** - opis wszystkich tokenów i cech używanych w modelu
4. **Opis metodologii** - jak dane są przetwarzane, jak model jest trenowany

---

## 🔄 Kolejność Wykonania

Ten notebook powinien być uruchomiony **przed** treningiem modelu (notebook 04-05), ale może być również uruchomiony **po** treningu, aby wygenerować tabele do publikacji.

**Wymagane pliki:**
- `corpus_combined_*.txt` - połączone dane z Synthea i MIMIC (z notebooka 03) - **automatycznie wykrywany najnowszy plik**
- `vocab_mlm.json` - słownik tokenów (z notebooka 04) - opcjonalne

---

## 📊 Struktura Danych

Model BERT klasyfikuje pacjentów **wieloetykietowo** do **8 klas** (zgodnie z `label_mapping.py`):
- **POZ** (Primary Care), **GASTROLOG**, **HEMATOLOG**, **NEFROLOG**, **SOR**, **KARDIOLOG**, **PULMONOLOG**, **HEPATOLOG**

Synthea (notebook 01) w tej wersji produkuje 5 klas (POZ, GASTROLOG, HEMATOLOG, NEFROLOG, SOR); MIMIC (notebook 02) produkuje wszystkie 8. Po merge corpus zawiera linie z 5 lub 8 etykietami; model i ewaluacja używają zawsze 8 wyjść.

**Format danych:** Każda linia w `corpus_combined_*.txt` to sekwencja tokenów + etykiety na końcu (multi-label: wiele TARGET_* oddzielonych spacją):
```
HGB_Q5 WBC_Q7 AGE_under_60 GENDER_F ... TARGET_POZ TARGET_SPEC_GASTROLOG
```


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import glob
import warnings
warnings.filterwarnings('ignore')

# ⚠️ AUTOMATYCZNE WYKRYWANIE NAJNOWSZEGO PLIKU
# Znajdź najnowszy corpus_combined
corpus_files = sorted(glob.glob("corpus_combined_*.txt"), reverse=True)
if not corpus_files:
    raise FileNotFoundError("Nie znaleziono pliku corpus_combined_*.txt. Uruchom najpierw notebook 03_Merge_Datasets.ipynb")
CORPUS_FILE = corpus_files[0]  # Najnowszy plik
print(f"📂 Używam pliku: {CORPUS_FILE}")

VOCAB_FILE = "vocab_mlm.json"  # Opcjonalne

# Ustawienia wizualizacji
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

print("📦 Biblioteki załadowane")
print(f"📂 Katalog roboczy: {Path.cwd()}")


## 1. Wczytanie i Analiza Danych

### 1.1. Wczytanie corpus_combined_*.txt


In [ ]:
# Wczytanie danych
# ⚠️ CORPUS_FILE jest już automatycznie wykryty w Cell 1
if not Path(CORPUS_FILE).exists():
    raise FileNotFoundError(
        f"❌ Plik {CORPUS_FILE} nie istnieje!\n"
        "   Uruchom najpierw notebook 03_Merge_Datasets.ipynb"
    )

print(f"📂 Wczytywanie {CORPUS_FILE}...")

texts = []
labels = []
all_tokens = []

with open(CORPUS_FILE, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        
        parts = line.split()
        if len(parts) < 2:
            continue
        
        # Ostatni element to label
        label_str = parts[-1]
        # Reszta to tokeny
        tokens = parts[:-1]
        
        texts.append(" ".join(tokens))
        labels.append(label_str)
        all_tokens.extend(tokens)
        
        if line_num % 50000 == 0:
            print(f"   Przetworzono {line_num:,} linii...")

print(f"\n✅ Załadowano {len(texts):,} próbek")
print(f"   Unikalnych tokenów: {len(set(all_tokens)):,}")
print(f"   Unikalnych labeli: {len(set(labels)):,}")

# Rozkład klas
label_counts = Counter(labels)
print("\n📊 Rozkład klas:")
for label, count in sorted(label_counts.items()):
    pct = count / len(labels) * 100
    print(f"   {label:25s}: {count:7,} ({pct:5.1f}%)")


## 2. Table 1: Charakterystyka Danych

### 2.1. Analiza tokenów i cech


In [ ]:
# ⚠️ WAŻNE: Ta komórka wymaga, aby Cell 3 została wykonana pomyślnie
# Sprawdzenie, czy dane zostały załadowane
if 'texts' not in globals() or 'labels' not in globals():
    raise NameError(
        "❌ Zmienne 'texts' i 'labels' nie są zdefiniowane!\n"
        "   Uruchom najpierw Cell 3 (wczytanie danych)."
    )

# Analiza tokenów - ekstrakcja cech
def extract_features_from_tokens(tokens):
    """Ekstrahuje cechy z tokenów (wiek, płeć, parametry laboratoryjne)"""
    features = {
        'age_group': None,
        'gender': None,
        'lab_tests': []
    }
    
    for token in tokens:
        if token.startswith('AGE_'):
            features['age_group'] = token.replace('AGE_', '')
        elif token.startswith('GENDER_'):
            features['gender'] = token.replace('GENDER_', '')
        elif '_Q' in token or '_VAL' in token:
            # Parametr laboratoryjny (np. HGB_Q5, WBC_Q7)
            lab_name = token.split('_')[0]
            features['lab_tests'].append(lab_name)
    
    return features

# Analiza dla każdej klasy
print("🔬 Analiza cech dla każdej klasy...\n")

class_features = {}
for label in set(labels):
    class_features[label] = {
        'age_groups': Counter(),
        'genders': Counter(),
        'lab_tests': Counter(),
        'total': 0
    }

# Przetwarzanie
for i, (text, label) in enumerate(zip(texts, labels)):
    tokens = text.split()
    features = extract_features_from_tokens(tokens)
    
    if features['age_group']:
        class_features[label]['age_groups'][features['age_group']] += 1
    if features['gender']:
        class_features[label]['genders'][features['gender']] += 1
    for lab in features['lab_tests']:
        class_features[label]['lab_tests'][lab] += 1
    class_features[label]['total'] += 1
    
    if (i + 1) % 50000 == 0:
        print(f"   Przetworzono {i+1:,} próbek...")

print("✅ Analiza zakończona")


### 2.2. Generowanie Table 1


In [ ]:
# ⚠️ WAŻNE: Ta komórka wymaga, aby Cell 3 i Cell 5 zostały wykonane pomyślnie
# Sprawdzenie, czy dane zostały załadowane i przetworzone
if 'labels' not in globals() or 'class_features' not in globals():
    raise NameError(
        "❌ Zmienne 'labels' i 'class_features' nie są zdefiniowane!\n"
        "   Uruchom najpierw Cell 3 (wczytanie danych) i Cell 5 (analiza cech)."
    )

# Mapowanie labeli na czytelne nazwy
LABEL_NAMES = {
    'TARGET_POZ': 'POZ',
    'TARGET_SPEC_GASTROLOG': 'Gastroenterologia',
    'TARGET_SPEC_HEMATOLOG': 'Hematologia',
    'TARGET_SPEC_NEFROLOG': 'Nefrologia',
    'TARGET_SPEC_SOR': 'SOR'
}

# Tworzenie Table 1
table1_data = []

for label in sorted(set(labels)):
    label_name = LABEL_NAMES.get(label, label)
    stats = class_features[label]
    total = stats['total']
    total_all = len(labels)
    
    # Rozkład wieku
    age_dist = stats['age_groups']
    age_str = ", ".join([f"{k}: {v} ({v/total*100:.1f}%)" for k, v in sorted(age_dist.items())[:3]]) if total > 0 else "N/A"
    
    # Rozkład płci
    gender_dist = stats['genders']
    gender_str = ", ".join([f"{k}: {v} ({v/total*100:.1f}%)" for k, v in sorted(gender_dist.items())]) if total > 0 else "N/A"
    
    # Najczęstsze parametry laboratoryjne
    top_labs = stats['lab_tests'].most_common(5)
    labs_str = ", ".join([f"{lab} ({count})" for lab, count in top_labs]) if top_labs else "N/A"
    
    table1_data.append({
        'Klasa': label_name,
        'N': f"{total:,}",
        '%': f"{total/total_all*100:.1f}%" if total_all > 0 else "0%",
        'Wiek': age_str,
        'Płeć': gender_str,
        'Parametry lab.': labs_str
    })

df_table1 = pd.DataFrame(table1_data)
print("📊 Table 1: Charakterystyka danych")
print("=" * 80)
display(df_table1)

# Zapis do CSV (dla publikacji)
df_table1.to_csv("table1_characteristics.csv", index=False, encoding='utf-8-sig')
print("\n✅ Table 1 zapisana do: table1_characteristics.csv")
